# 10 — Getting started

The motivation behind scikit-ops is to allow very repeatable, very precise,
isolated environments to hold dependencies that potentially conflict with the
host environment — for example, a dependency that can't be installed alongside
the latest version of napari. The scikit-ops framework, via appose, takes care
of the details of that installation. The host dependencies are simple and easy for 
the user to install.

Here we install the host, run one op, check the GPU, and fetch the images.

## Install

Start from a clean environment. Python 3.12 or newer.

```sh
conda create -n i2k2026 python=3.12
conda activate i2k2026
```

Then the host — notebook kernel and napari, no torch, no TensorFlow, no CUDA.

```sh
pip install "scikit-ops @ git+https://github.com/apposed/scikit-ops.git"
pip install napari-ai-lab "napari[all]" "tnia-python[plotting]" jupyterlab matplotlib scikit-image tifffile
```

In [1]:
import skop
import napari_ai_lab

print('skop         ', skop.__file__)
print('napari_ai_lab', napari_ai_lab.__file__)

skop          /home/bnorthan/mambaforge/envs/i2k2026/lib/python3.12/site-packages/skop/__init__.py
napari_ai_lab /home/bnorthan/mambaforge/envs/i2k2026/lib/python3.12/site-packages/napari_ai_lab/__init__.py


## Does an op run, and does it report back?

`slow_sum` runs in the light `minimal` env and emits four progress events.

In [2]:
import numpy as np
from skop.runner import Runner
from skop.ops.toy import slow_sum

runner = Runner()

def show(ev):
    print('EVENT', ev.current, ev.maximum, ev.message, flush=True)

total = runner.run(slow_sum, image=np.ones((4, 4), np.float32), steps=4,
                   on_progress=show)
print('result:', total)

EVENT None None None
EVENT 0 4 Summing chunk 1 of 4
EVENT 1 4 Summing chunk 2 of 4
EVENT 2 4 Summing chunk 3 of 4
EVENT 3 4 Summing chunk 4 of 4
EVENT None None None
result: 16.0


## The StarDist environment

`up to date` returns at once. `stale` or `missing` rebuilds — minutes and
gigabytes, so do it before the session.

In [3]:
print('status:', runner.environment_status('stardist-tf'))
runner.ensure_environment('stardist-tf')

status: up to date
stardist-tf: up to date


<appose.builder.BaseBuilder._create_env.<locals>.BuiltEnvironment at 0x719c1924bfb0>

## Is the GPU visible?

TensorFlow reports missing CUDA as a warning and carries on with the CPU, so
ask it directly.

In [4]:
# Through `pixi run`: calling the env's python directly skips activation and
# reports no GPU on a working install.
manifest = runner.env_dir('stardist-tf') / 'pixi.toml'

!pixi run --manifest-path {manifest} python -c "import tensorflow as tf; print('GPUs:', tf.config.list_physical_devices('GPU'))" 2>&1 | grep -Ei 'gpus:|could not load'

GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## Get the images

Seven frames of bees on honeycomb, with labels. 32 MB.

In [5]:
import urllib.request
import zipfile
from pathlib import Path

URL = 'https://www.dropbox.com/scl/fo/ock3qfwx11mvbfuih0hsc/AGk9LFOqnfm3HG_zw22vIv8?rlkey=aaoac1euyg89hpadnms96e2c9&st=4ck6sxe3&dl=1'   # must end in ?dl=1
DATA = Path('data')

DATA.mkdir(exist_ok=True)
if not (DATA / 'bees').exists():
    urllib.request.urlretrieve(URL, DATA / 'bees.zip')
    zipfile.ZipFile(DATA / 'bees.zip').extractall(DATA)

print(sorted(p.name for p in (DATA / 'bees').glob('*.png')))

['comb_01.png', 'comb_02.png', 'comb_03.png', 'comb_04.png', 'comb_05.png', 'comb_06.png', 'comb_07.png']


## Load one back

Check the files are where we think they are.

In [ ]:
from skimage.io import imread
import matplotlib.pyplot as plt

img = imread(DATA / 'bees' / 'comb_03.png')
print(img.shape, img.dtype)

plt.imshow(img)
plt.axis('off')